# Back-translation quality QA

Round-trip check for forward Google Translate: `lang_content` → English (`back_en`) vs original `en_content`.

**Metrics:** chrF++ (primary lexical), BLEU (secondary), embedding cosine (semantic), length ratio.

**Caveats:** RTT is a proxy (same engine forward/back can cancel errors). Prefer **relative** language ranks over absolute cosine cutoffs.

**Data:** Prefer `data/dataset_v1/backtranslate_scored.parquet` (full Sheets fills). Falls back to smoke scores if the full file is missing.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
if not (ROOT / "data" / "dataset_v1").exists():
    ROOT = Path(".").resolve()

SCORED = ROOT / "data" / "dataset_v1" / "backtranslate_scored.parquet"
SMOKE = ROOT / "data" / "dataset_v1" / "backtranslate_scored_smoke.parquet"
SUMMARY = ROOT / "data" / "dataset_v1" / "backtranslate_lang_summary.csv"

path = SCORED if SCORED.exists() else SMOKE
assert path.exists(), f"Missing scored parquet. Run --score after Sheets fills.\nLooked for {SCORED} and {SMOKE}"
df = pd.read_parquet(path)
print("loaded", path.relative_to(ROOT), "rows", len(df), "langs", sorted(df.lang.unique()))
df.head(2)


In [ ]:
# Per-language / bracket summary
summary = (
    df.groupby(["lang", "resource_bracket"], as_index=False)
    .agg(
        n=("content_id", "count"),
        chrf_mean=("chrf", "mean"),
        chrf_median=("chrf", "median"),
        chrf_p10=("chrf", lambda s: float(s.quantile(0.10))),
        bleu_mean=("bleu", "mean"),
        embed_cosine_mean=("embed_cosine", "mean"),
        embed_cosine_median=("embed_cosine", "median"),
        embed_cosine_p10=("embed_cosine", lambda s: float(s.quantile(0.10))),
        len_ratio_mean=("len_ratio", "mean"),
    )
    .sort_values("chrf_mean", ascending=False)
)
display(summary.round(4))
if SUMMARY.exists():
    print("Also on disk:", SUMMARY.relative_to(ROOT))


In [ ]:
# Bar: langs sorted by mean chrF++
plot_df = summary.sort_values("chrf_mean")
colors = [
    "#c44e52" if L in {"yo", "sw", "ne"} else "#4c72b0"
    for L in plot_df.lang
]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(plot_df.lang, plot_df.chrf_mean, color=colors)
ax.set_xlabel("chrF++ mean (0–100)")
ax.set_title("Round-trip chrF++ by language (low bracket in red)")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter: lexical vs semantic agreement
fig, ax = plt.subplots(figsize=(6, 5))
for bracket, g in df.groupby("resource_bracket"):
    ax.scatter(g.chrf, g.embed_cosine, s=12, alpha=0.35, label=bracket)
ax.set_xlabel("chrF++")
ax.set_ylabel("embed_cosine")
ax.set_title("chrF++ vs embedding cosine")
ax.legend(title="bracket", markerscale=2)
plt.tight_layout()
plt.show()


In [ ]:
# Worst rows overall (lowest chrF++)
worst = df.nsmallest(20, "chrf")[
    ["content_id", "lang", "label", "chrf", "bleu", "embed_cosine", "en_content", "back_en"]
]
display(worst)


In [ ]:
# Worst 10 per low-resource language
low = df[df.lang.isin(["yo", "sw", "ne"])]
for lang, g in low.groupby("lang"):
    print("===", lang, "===")
    display(
        g.nsmallest(10, "chrf")[
            ["content_id", "chrf", "embed_cosine", "en_content", "back_en", "lang_content"]
        ]
    )


## How to refresh after Sheets fills

```bash
# Upload backtranslate.xlsx → Google Sheets; wait for every language tab.
# Download as backtranslate_filled.xlsx into:
#   data/dataset_v1/sheets/backtranslate/

.venv/bin/python scripts/prepare_backtranslate_qa.py --score \
  --filled data/dataset_v1/sheets/backtranslate/backtranslate_filled.xlsx \
  --out data/dataset_v1/backtranslate_scored.parquet
```

Then re-run this notebook (it will prefer the full scored file).
